In [1]:
from pathlib import Path
import json
import platform
import sys
from datetime import datetime

# ============================================================
# 1. Project paths
# ============================================================

PROJECT_ROOT = Path.home() / "Desktop" / "qskit" / "qec_ml"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

PHASE2_FREEZE_DIR = NOTEBOOKS_DIR / "phase2_freeze_20261104"
PHASE2_FREEZE_ZIP = NOTEBOOKS_DIR / "PHASE2_FREEZE_20261104.zip"

# Future output directory for Phase 4.
# Creating this directory does not modify the Phase 2 freeze.
PHASE4_OUTPUT_DIR = NOTEBOOKS_DIR / "phase4_final_test_results"

# ============================================================
# 2. Frozen experiment configuration
# ============================================================

EXPERIMENT_CONFIG = {
    "project_phase": "Phase 4 — final held-out test evaluation",

    # Quantum error-correction setup
    "code_family": "rotated surface code",
    "code_distance": 3,
    "rounds": 3,
    "physical_error_probability": 0.01,
    "stim_circuit": "surface_code:rotated_memory_x",

    # Original dataset split
    "total_shots": 20000,
    "training_samples": 13999,
    "validation_samples": 2999,
    "test_samples": 3002,

    # Frozen evaluation protocol
    "rl_evaluation_policy": "greedy",
    "rl_replicates_per_strategy": 8,
    "rl_strategies": [
        "natural",
        "80_20",
        "75_25",
        "70_30",
        "65_35",
        "60_40",
        "50_50",
    ],

    # Baselines
    "baseline_methods": [
        "PyMatching",
        "Supervised MLP",
    ],

    # Test-set safeguard
    "test_set_used_for_model_selection": False,
    "retraining_allowed": False,
    "hyperparameter_tuning_allowed": False,
}

# ============================================================
# 3. Create Phase 4 output directory
# ============================================================

PHASE4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 4. Record the evaluation session
# ============================================================

SESSION_INFO = {
    "created_at": datetime.now().astimezone().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "project_root": str(PROJECT_ROOT),
    "phase2_freeze_directory": str(PHASE2_FREEZE_DIR),
    "phase2_freeze_zip": str(PHASE2_FREEZE_ZIP),
    "phase4_output_directory": str(PHASE4_OUTPUT_DIR),
}

# ============================================================
# 5. Check that the Phase 2 freeze is available
# ============================================================

print("=" * 65)
print("PHASE 4 — CONFIGURATION AND PROVENANCE")
print("=" * 65)

print("\nProject paths:")
print(f"Project root       : {PROJECT_ROOT}")
print(f"Project exists     : {PROJECT_ROOT.exists()}")
print(f"Phase 2 freeze dir : {PHASE2_FREEZE_DIR}")
print(f"Freeze dir exists  : {PHASE2_FREEZE_DIR.is_dir()}")
print(f"Freeze ZIP         : {PHASE2_FREEZE_ZIP}")
print(f"Freeze ZIP exists  : {PHASE2_FREEZE_ZIP.is_file()}")
print(f"Phase 4 output     : {PHASE4_OUTPUT_DIR}")

print("\nFrozen experiment configuration:")
for key, value in EXPERIMENT_CONFIG.items():
    print(f"{key:40s}: {value}")

print("\nSession information:")
for key, value in SESSION_INFO.items():
    print(f"{key:40s}: {value}")

# ============================================================
# 6. Read only freeze metadata (if available)
# ============================================================

metadata_path = PHASE2_FREEZE_DIR / "freeze_metadata.json"

if metadata_path.is_file():
    with metadata_path.open("r", encoding="utf-8") as f:
        FREEZE_METADATA = json.load(f)

    print("\nPhase 2 freeze metadata:")
    print(json.dumps(FREEZE_METADATA, indent=2))

else:
    FREEZE_METADATA = None
    print("\nWARNING: freeze_metadata.json was not found.")
    print("We will investigate the freeze structure before proceeding.")

# ============================================================
# 7. Basic configuration checks
# ============================================================

assert EXPERIMENT_CONFIG["training_samples"] + \
       EXPERIMENT_CONFIG["validation_samples"] + \
       EXPERIMENT_CONFIG["test_samples"] == \
       EXPERIMENT_CONFIG["total_shots"], \
       "Dataset split counts do not sum to the total shot count."

assert EXPERIMENT_CONFIG["test_set_used_for_model_selection"] is False
assert EXPERIMENT_CONFIG["retraining_allowed"] is False
assert EXPERIMENT_CONFIG["hyperparameter_tuning_allowed"] is False

print("\nConfiguration checks: PASS")
print("No test samples have been loaded or evaluated.")

PHASE 4 — CONFIGURATION AND PROVENANCE

Project paths:
Project root       : /home/ankit/Desktop/qskit/qec_ml
Project exists     : True
Phase 2 freeze dir : /home/ankit/Desktop/qskit/qec_ml/notebooks/phase2_freeze_20261104
Freeze dir exists  : True
Freeze ZIP         : /home/ankit/Desktop/qskit/qec_ml/notebooks/PHASE2_FREEZE_20261104.zip
Freeze ZIP exists  : True
Phase 4 output     : /home/ankit/Desktop/qskit/qec_ml/notebooks/phase4_final_test_results

Frozen experiment configuration:
project_phase                           : Phase 4 — final held-out test evaluation
code_family                             : rotated surface code
code_distance                           : 3
rounds                                  : 3
physical_error_probability              : 0.01
stim_circuit                            : surface_code:rotated_memory_x
total_shots                             : 20000
training_samples                        : 13999
validation_samples                      : 2999
test_samples   

In [2]:
from pathlib import Path
import json
import csv

# ============================================================
# 1. Resolve source RL archive from frozen metadata
# ============================================================

assert FREEZE_METADATA is not None, (
    "Freeze metadata is unavailable. Cannot safely audit provenance."
)

source_rl_archive = Path(
    FREEZE_METADATA["source_rl_archive"]
)

print("=" * 75)
print("PHASE 2 ARTIFACT AUDIT")
print("=" * 75)

print("\nSource RL archive:")
print(source_rl_archive)
print("Exists:", source_rl_archive.is_dir())

# ============================================================
# 2. Inventory freeze directory
# ============================================================

def inventory_directory(root):
    """Return a sorted inventory of files under a directory."""
    root = Path(root)
    records = []

    if not root.is_dir():
        return records

    for path in sorted(root.rglob("*")):
        if path.is_file():
            records.append({
                "relative_path": str(path.relative_to(root)),
                "absolute_path": str(path),
                "size_bytes": path.stat().st_size,
                "suffix": path.suffix.lower(),
            })

    return records


freeze_inventory = inventory_directory(PHASE2_FREEZE_DIR)
rl_inventory = inventory_directory(source_rl_archive)

print("\nFreeze directory inventory")
print("-" * 75)
print("Files:", len(freeze_inventory))

for item in freeze_inventory:
    print(
        f'{item["size_bytes"]:>12,} bytes  '
        f'{item["relative_path"]}'
    )

print("\nOriginal RL archive inventory")
print("-" * 75)

if source_rl_archive.is_dir():
    print("Files:", len(rl_inventory))

    for item in rl_inventory:
        print(
            f'{item["size_bytes"]:>12,} bytes  '
            f'{item["relative_path"]}'
        )
else:
    print("WARNING: Original RL archive directory is unavailable.")

# ============================================================
# 3. Identify model/checkpoint candidates
# ============================================================

checkpoint_suffixes = {
    ".pt", ".pth", ".ckpt", ".pkl", ".pickle",
    ".joblib", ".safetensors"
}

def is_checkpoint_candidate(item):
    path = Path(item["relative_path"])
    name = path.name.lower()

    return (
        path.suffix.lower() in checkpoint_suffixes
        or "checkpoint" in name
        or "model_weights" in name
    )


freeze_checkpoints = [
    item for item in freeze_inventory
    if is_checkpoint_candidate(item)
]

rl_checkpoints = [
    item for item in rl_inventory
    if is_checkpoint_candidate(item)
]

print("\nCheckpoint candidates in freeze directory")
print("-" * 75)

if freeze_checkpoints:
    for item in freeze_checkpoints:
        print(
            f'{item["size_bytes"]:>12,} bytes  '
            f'{item["relative_path"]}'
        )
else:
    print("No checkpoint candidates found.")

print("\nCheckpoint candidates in original RL archive")
print("-" * 75)

if rl_checkpoints:
    for item in rl_checkpoints:
        print(
            f'{item["size_bytes"]:>12,} bytes  '
            f'{item["relative_path"]}'
        )
else:
    print("No checkpoint candidates found.")

# ============================================================
# 4. Inspect manifest entries
# ============================================================

manifest_path = PHASE2_FREEZE_DIR / "manifest_sha256.csv"

print("\nFreeze manifest")
print("-" * 75)

manifest_records = []

if manifest_path.is_file():
    with manifest_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        manifest_records = list(reader)

    print("Manifest entries:", len(manifest_records))
    print("Manifest columns:", reader.fieldnames)

    for record in manifest_records:
        print(record)
else:
    print("WARNING: manifest_sha256.csv was not found.")

# ============================================================
# 5. Summarize audit findings
# ============================================================

print("\nAudit summary")
print("-" * 75)

print("Freeze directory available:", PHASE2_FREEZE_DIR.is_dir())
print("Original RL archive available:", source_rl_archive.is_dir())
print("Freeze files:", len(freeze_inventory))
print("Original RL archive files:", len(rl_inventory))
print("Freeze checkpoint candidates:", len(freeze_checkpoints))
print("RL archive checkpoint candidates:", len(rl_checkpoints))
print("Manifest entries:", len(manifest_records))

print("\nBaseline checkpoint candidates:")

baseline_candidates = [
    item for item in freeze_inventory
    if any(
        term in item["relative_path"].lower()
        for term in ["pymatching", "supervised_mlp", "mlp_decoder"]
    )
]

if baseline_candidates:
    for item in baseline_candidates:
        print(
            f'{item["size_bytes"]:>12,} bytes  '
            f'{item["relative_path"]}'
        )
else:
    print("No baseline-specific files identified by filename.")

print("\nNo model weights were loaded.")
print("No training or evaluation was performed.")
print("No test samples were accessed.")

PHASE 2 ARTIFACT AUDIT

Source RL archive:
/home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints/paired_multiseed_fixed_budget_20261101_154138/validation_evaluation_56_models_20261101_171108
Exists: True

Freeze directory inventory
---------------------------------------------------------------------------
Files: 50
       5,795 bytes  PHASE2_FINAL_REPORT.md
         198 bytes  baselines/pymatching_validation_by_class.csv
         224 bytes  baselines/pymatching_validation_overall.csv
      34,365 bytes  baselines/pymatching_validation_predictions.csv
      82,526 bytes  baselines/supervised_mlp_decoder.pt
       2,053 bytes  baselines/supervised_mlp_training_history.csv
         196 bytes  baselines/supervised_mlp_validation_by_class.csv
         283 bytes  baselines/supervised_mlp_validation_overall.csv
      70,266 bytes  baselines/supervised_mlp_validation_predictions.csv
       1,421 bytes  freeze_metadata.json
       7,471 bytes  manifest_sha256.csv
       8,264 bytes 

In [3]:
from pathlib import Path
import json

# ============================================================
# 1. Define checkpoint search locations
# ============================================================

CHECKPOINT_ROOT = NOTEBOOKS_DIR / "qec_model_checkpoints"

checkpoint_extensions = {
    ".pt", ".pth", ".ckpt", ".pkl",
    ".pickle", ".joblib", ".safetensors"
}

print("=" * 75)
print("SEARCHING FOR SEPARATELY SAVED RL CHECKPOINTS")
print("=" * 75)

print("\nCheckpoint root:", CHECKPOINT_ROOT)
print("Exists:", CHECKPOINT_ROOT.is_dir())

# ============================================================
# 2. Find all checkpoint-like files
# ============================================================

checkpoint_files = []

if CHECKPOINT_ROOT.is_dir():
    for path in CHECKPOINT_ROOT.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()

        if (
            path.suffix.lower() in checkpoint_extensions
            or "checkpoint" in name
            or "model_weights" in name
        ):
            checkpoint_files.append({
                "path": path,
                "size_bytes": path.stat().st_size,
                "modified": path.stat().st_mtime,
            })

checkpoint_files.sort(key=lambda x: str(x["path"]))

print("\nCheckpoint candidates found:", len(checkpoint_files))
print("-" * 75)

for item in checkpoint_files:
    print(f'{item["size_bytes"]:>12,} bytes  {item["path"]}')

# ============================================================
# 3. Inspect RL evaluation metadata for saved-model references
# ============================================================

metadata_files = [
    source_rl_archive / "evaluation_metadata.json",
    source_rl_archive / "evaluation_progress.json",
]

search_terms = (
    "checkpoint",
    "model_path",
    "policy_path",
    "weights_path",
    "state_dict",
    "saved_model",
)

def find_references(obj, prefix=""):
    """Find metadata fields whose names suggest model/checkpoint paths."""
    matches = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            current = f"{prefix}.{key}" if prefix else key

            if any(term in key.lower() for term in search_terms):
                matches.append((current, value))

            matches.extend(find_references(value, current))

    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            matches.extend(
                find_references(value, f"{prefix}[{index}]")
            )

    return matches


print("\nMetadata checkpoint references")
print("-" * 75)

for metadata_file in metadata_files:
    print(f"\nFile: {metadata_file.name}")

    if not metadata_file.is_file():
        print("Not found.")
        continue

    with metadata_file.open("r", encoding="utf-8") as f:
        metadata = json.load(f)

    references = find_references(metadata)

    if references:
        for key, value in references:
            print(f"{key}: {value}")
    else:
        print("No explicit checkpoint/model path fields found.")

# ============================================================
# 4. Inspect model-related source/configuration files
# ============================================================

print("\nModel-related files under notebooks")
print("-" * 75)

source_candidates = []

for path in NOTEBOOKS_DIR.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if (
        path.suffix.lower() in {".py", ".json", ".yaml", ".yml"}
        and any(term in name for term in (
            "train", "config", "model", "experiment"
        ))
    ):
        source_candidates.append(path)

for path in sorted(source_candidates):
    print(path)

# ============================================================
# 5. Summary
# ============================================================

print("\n" + "=" * 75)
print("SEARCH SUMMARY")
print("=" * 75)

print("Checkpoint root exists:", CHECKPOINT_ROOT.is_dir())
print("Checkpoint candidates:", len(checkpoint_files))
print("Metadata files inspected:", len(metadata_files))
print("Model/config source files:", len(source_candidates))

if checkpoint_files:
    print("\nPotential RL checkpoints were found.")
    print("We must identify their strategy and replicate provenance next.")
else:
    print("\nNo separately saved checkpoint candidates were found.")
    print("Do not begin final RL test evaluation yet.")

print("\nNo checkpoint was loaded.")
print("No training or evaluation was performed.")
print("No test samples were accessed.")

SEARCHING FOR SEPARATELY SAVED RL CHECKPOINTS

Checkpoint root: /home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints
Exists: True

Checkpoint candidates found: 83
---------------------------------------------------------------------------
     476,274 bytes  /home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints/fixed_budget_sampling_trials_20261031_030151/50_50_fixed_budget.pt
     478,514 bytes  /home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints/fixed_budget_sampling_trials_20261031_030151/60_40_fixed_budget.pt
     479,474 bytes  /home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints/fixed_budget_sampling_trials_20261031_030151/65_35_fixed_budget.pt
     480,626 bytes  /home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints/fixed_budget_sampling_trials_20261031_030151/70_30_fixed_budget.pt
     489,266 bytes  /home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints/fixed_budget_sampling_trials_20261031_030151/75_25_fixed

In [4]:
from pathlib import Path
import json
import hashlib
import re
import pandas as pd

# ============================================================
# 1. Define the authoritative multi-seed experiment directory
# ============================================================

RL_EXPERIMENT_DIR = (
    CHECKPOINT_ROOT
    / "paired_multiseed_fixed_budget_20261101_154138"
)

RL_CHECKPOINT_DIR_1_3 = RL_EXPERIMENT_DIR / "checkpoints"

RL_EXTENSION_DIR = RL_EXPERIMENT_DIR / "extension_replicates_04_08"

RL_CHECKPOINT_DIR_4_8 = RL_EXTENSION_DIR / "checkpoints"

RL_CONFIG_PATH = RL_EXPERIMENT_DIR / "experiment_config.json"
RL_TRAINING_PROGRESS_PATH = RL_EXPERIMENT_DIR / "training_progress.json"

RL_EXTENSION_CONFIG_PATH = RL_EXTENSION_DIR / "experiment_config.json"
RL_EXTENSION_PROGRESS_PATH = RL_EXTENSION_DIR / "training_progress.json"

print("=" * 75)
print("PAIRED MULTI-SEED RL CHECKPOINT VERIFICATION")
print("=" * 75)

for label, path in [
    ("Experiment directory", RL_EXPERIMENT_DIR),
    ("Replicate 1–3 checkpoint directory", RL_CHECKPOINT_DIR_1_3),
    ("Replicate 4–8 checkpoint directory", RL_CHECKPOINT_DIR_4_8),
    ("Main experiment config", RL_CONFIG_PATH),
    ("Main training progress", RL_TRAINING_PROGRESS_PATH),
    ("Extension experiment config", RL_EXTENSION_CONFIG_PATH),
    ("Extension training progress", RL_EXTENSION_PROGRESS_PATH),
]:
    print(f"{label:40s}: {path.exists()}")

assert RL_EXPERIMENT_DIR.is_dir()
assert RL_CHECKPOINT_DIR_1_3.is_dir()
assert RL_CHECKPOINT_DIR_4_8.is_dir()

# ============================================================
# 2. Read experiment configuration and training progress
# ============================================================

def read_json(path):
    if not path.is_file():
        return None

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


main_config = read_json(RL_CONFIG_PATH)
main_progress = read_json(RL_TRAINING_PROGRESS_PATH)

extension_config = read_json(RL_EXTENSION_CONFIG_PATH)
extension_progress = read_json(RL_EXTENSION_PROGRESS_PATH)

print("\nMAIN EXPERIMENT CONFIGURATION")
print("-" * 75)
print(json.dumps(main_config, indent=2))

print("\nMAIN TRAINING PROGRESS")
print("-" * 75)
print(json.dumps(main_progress, indent=2))

print("\nEXTENSION EXPERIMENT CONFIGURATION")
print("-" * 75)
print(json.dumps(extension_config, indent=2))

print("\nEXTENSION TRAINING PROGRESS")
print("-" * 75)
print(json.dumps(extension_progress, indent=2))

# ============================================================
# 3. Define the expected strategy and replicate combinations
# ============================================================

EXPECTED_STRATEGIES = [
    "natural",
    "80_20",
    "75_25",
    "70_30",
    "65_35",
    "60_40",
    "50_50",
]

EXPECTED_REPLICATES = range(1, 9)

def expected_checkpoint_name(replicate, strategy):
    return f"replicate_{replicate:02d}_{strategy}.pt"


# ============================================================
# 4. Build the expected checkpoint map
# ============================================================

checkpoint_records = []

for replicate in EXPECTED_REPLICATES:
    if replicate <= 3:
        checkpoint_dir = RL_CHECKPOINT_DIR_1_3
    else:
        checkpoint_dir = RL_CHECKPOINT_DIR_4_8

    for strategy in EXPECTED_STRATEGIES:
        filename = expected_checkpoint_name(replicate, strategy)
        path = checkpoint_dir / filename

        checkpoint_records.append({
            "replicate": replicate,
            "strategy": strategy,
            "filename": filename,
            "path": path,
            "exists": path.is_file(),
            "size_bytes": path.stat().st_size if path.is_file() else None,
        })

checkpoint_df = pd.DataFrame(checkpoint_records)

print("\nEXPECTED CHECKPOINT MATRIX")
print("-" * 75)

checkpoint_matrix = checkpoint_df.pivot(
    index="replicate",
    columns="strategy",
    values="exists"
)

display(checkpoint_matrix)

print("\nCheckpoint counts:")
print("Expected:", len(checkpoint_records))
print("Found:", int(checkpoint_df["exists"].sum()))
print("Missing:", int((~checkpoint_df["exists"]).sum()))

# ============================================================
# 5. Calculate SHA-256 hashes for all found checkpoints
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


for index, row in checkpoint_df.iterrows():
    if row["exists"]:
        checkpoint_df.loc[index, "sha256"] = sha256_file(row["path"])
    else:
        checkpoint_df.loc[index, "sha256"] = None

print("\nCHECKPOINT INVENTORY")
print("-" * 75)

display(
    checkpoint_df[
        [
            "replicate",
            "strategy",
            "filename",
            "exists",
            "size_bytes",
            "sha256",
        ]
    ]
)

# ============================================================
# 6. Check for duplicate checkpoint hashes
# ============================================================

available_hashes = checkpoint_df.loc[
    checkpoint_df["exists"], "sha256"
]

duplicate_hashes = available_hashes[
    available_hashes.duplicated(keep=False)
]

print("\nDuplicate checkpoint hashes:")
if duplicate_hashes.empty:
    print("None.")
else:
    duplicate_rows = checkpoint_df[
        checkpoint_df["sha256"].isin(duplicate_hashes)
    ]
    display(duplicate_rows[
        ["replicate", "strategy", "filename", "sha256"]
    ])

# ============================================================
# 7. Validate expected count and file completeness
# ============================================================

all_checkpoints_present = bool(checkpoint_df["exists"].all())

assert len(checkpoint_df) == 56
assert all_checkpoints_present, (
    "Some expected checkpoint files are missing. "
    "Do not proceed to model loading."
)

print("\n" + "=" * 75)
print("CHECKPOINT FILE VERIFICATION: PASS")
print("=" * 75)

print("56 expected checkpoint files found.")
print("All seven strategies are represented in all eight replicates.")
print("Checkpoint hashes calculated.")
print("No model weights loaded.")
print("No training or evaluation performed.")
print("No test samples accessed.")

PAIRED MULTI-SEED RL CHECKPOINT VERIFICATION
Experiment directory                    : True
Replicate 1–3 checkpoint directory      : True
Replicate 4–8 checkpoint directory      : True
Main experiment config                  : True
Main training progress                  : True
Extension experiment config             : True
Extension training progress             : True

MAIN EXPERIMENT CONFIGURATION
---------------------------------------------------------------------------
{
  "experiment_name": "paired_multiseed_fixed_budget",
  "created_at": "2026-11-01T15:41:38.039672",
  "num_new_replicates": 3,
  "num_sampling_conditions": 7,
  "num_new_training_trials": 21,
  "architecture": {
    "observation_dim": 30,
    "hidden_dimensions": [
      128,
      128
    ],
    "action_dim": 10,
    "parameter_count": 21770
  },
  "training": {
    "gradient_update_budget": 7000,
    "target_update_interval": 200,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "epsilon_decay_updates": 

strategy,50_50,60_40,65_35,70_30,75_25,80_20,natural
replicate,,,,,,,
1,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True
5,True,True,True,True,True,True,True
6,True,True,True,True,True,True,True
7,True,True,True,True,True,True,True
8,True,True,True,True,True,True,True



Checkpoint counts:
Expected: 56
Found: 56
Missing: 0

CHECKPOINT INVENTORY
---------------------------------------------------------------------------


,replicate,strategy,filename,exists,size_bytes,sha256
0,1,natural,replicate_01_natural.pt,True,288755,e7100683a1b71e929852767bc5af6708e403ce95559808...
1,1,80_20,replicate_01_80_20.pt,True,289737,93b1cd4d20a667ef9716e7469f622be50fd8c8e879ad5d...
2,1,75_25,replicate_01_75_25.pt,True,288201,694ee4302acab9b6d57e5320ca101d71da18bcac5fc472...
3,1,70_30,replicate_01_70_30.pt,True,283017,5894ba32f236dff20c9e2e92f09237959b9251c57fe8c3...
4,1,65_35,replicate_01_65_35.pt,True,277961,a0f84dfaf445da35296962c6a4598249e912c289371861...
5,1,60_40,replicate_01_60_40.pt,True,281033,50dbb518b399a007dae9de074ae11313a8cb2a5a1d3b6c...
6,1,50_50,replicate_01_50_50.pt,True,272329,75e6421d9c95cbe32a3a45c47a06393edf2449ef1372c6...
7,2,natural,replicate_02_natural.pt,True,287027,a2849ba13a59fb20b7b71007d9b49a2501647c7e2992f5...
8,2,80_20,replicate_02_80_20.pt,True,285129,db1a205cef45fa65133702fff788f07b1c3cbf11d8cf83...
9,2,75_25,replicate_02_75_25.pt,True,285385,8673313e0601bf13759ce7b652b74c0b28fa5b2b693ca8...



Duplicate checkpoint hashes:
None.

CHECKPOINT FILE VERIFICATION: PASS
56 expected checkpoint files found.
All seven strategies are represented in all eight replicates.
Checkpoint hashes calculated.
No model weights loaded.
No training or evaluation performed.
No test samples accessed.


In [5]:
from pathlib import Path
import hashlib
import json
import torch

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

RL_EXPERIMENT_DIR = (
    PROJECT_ROOT
    / "notebooks"
    / "qec_model_checkpoints"
    / "paired_multiseed_fixed_budget_20261101_154138"
)

RL_EXTENSION_DIR = RL_EXPERIMENT_DIR / "extension_replicates_04_08"

RL_CHECKPOINT_DIR_1_3 = RL_EXPERIMENT_DIR / "checkpoints"
RL_CHECKPOINT_DIR_4_8 = RL_EXTENSION_DIR / "checkpoints"

MAIN_PROGRESS_PATH = RL_EXPERIMENT_DIR / "training_progress.json"
EXTENSION_PROGRESS_PATH = RL_EXTENSION_DIR / "training_progress.json"

STRATEGIES = [
    "natural",
    "80_20",
    "75_25",
    "70_30",
    "65_35",
    "60_40",
    "50_50",
]

# ------------------------------------------------------------
# 2. SHA-256 helper
# ------------------------------------------------------------

def calculate_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. Read recorded training-progress metadata
# ------------------------------------------------------------

with open(MAIN_PROGRESS_PATH, "r") as file:
    main_progress = json.load(file)

with open(EXTENSION_PROGRESS_PATH, "r") as file:
    extension_progress = json.load(file)

recorded_hashes = {}

# Replicates 1–3: dictionary keyed by trial name.
for trial_key, trial_info in main_progress["trials"].items():
    filename = trial_info["checkpoint"]

    replicate = int(trial_info["replicate"])
    strategy = trial_info["ratio_name"]

    checkpoint_path = RL_CHECKPOINT_DIR_1_3 / filename

    recorded_hashes[(replicate, strategy)] = {
        "path": checkpoint_path,
        "recorded_sha256": trial_info["sha256"],
    }

# Replicates 4–8: list of trial records.
for trial_info in extension_progress["trials"]:
    replicate = int(trial_info["replicate"])
    strategy = trial_info["ratio_name"]

    checkpoint_path = (
        RL_EXTENSION_DIR / trial_info["checkpoint_file"]
    )

    recorded_hashes[(replicate, strategy)] = {
        "path": checkpoint_path,
        "recorded_sha256": trial_info["checkpoint_sha256"],
    }


# ------------------------------------------------------------
# 4. Compare actual hashes with recorded hashes
# ------------------------------------------------------------

hash_verification = []

for replicate in range(1, 9):
    for strategy in STRATEGIES:

        key = (replicate, strategy)
        record = recorded_hashes.get(key)

        if record is None:
            hash_verification.append({
                "replicate": replicate,
                "strategy": strategy,
                "exists": False,
                "hash_matches": False,
                "status": "MISSING_METADATA",
            })
            continue

        checkpoint_path = record["path"]

        if not checkpoint_path.exists():
            hash_verification.append({
                "replicate": replicate,
                "strategy": strategy,
                "exists": False,
                "hash_matches": False,
                "status": "MISSING_FILE",
            })
            continue

        actual_hash = calculate_sha256(checkpoint_path)
        hash_matches = actual_hash == record["recorded_sha256"]

        hash_verification.append({
            "replicate": replicate,
            "strategy": strategy,
            "exists": True,
            "hash_matches": hash_matches,
            "status": "PASS" if hash_matches else "HASH_MISMATCH",
        })

import pandas as pd

hash_df = pd.DataFrame(hash_verification)

print("=" * 75)
print("CHECKPOINT HASH VERIFICATION")
print("=" * 75)

display(
    hash_df.pivot(
        index="replicate",
        columns="strategy",
        values="status"
    )
)

print("\nVerification summary:")
print(hash_df["status"].value_counts().to_string())

all_hashes_match = (
    len(hash_df) == 56
    and hash_df["exists"].all()
    and hash_df["hash_matches"].all()
)

print("\nAll 56 recorded hashes match:", all_hashes_match)


# ------------------------------------------------------------
# 5. Inspect checkpoint serialization safely
# ------------------------------------------------------------

# Inspect one checkpoint from the original experiment and one
# from the extension. Both are loaded on CPU.
SAMPLE_CHECKPOINTS = {
    "replicate_01_natural": (
        RL_CHECKPOINT_DIR_1_3 / "replicate_01_natural.pt"
    ),
    "replicate_04_natural": (
        RL_CHECKPOINT_DIR_4_8 / "replicate_04_natural.pt"
    ),
}


def describe_checkpoint_object(obj, prefix="root", depth=0, max_depth=3):
    """
    Print checkpoint structure without printing tensor values.
    """
    indent = "  " * depth

    if isinstance(obj, dict):
        print(f"{indent}{prefix}: dict with {len(obj)} entries")

        if depth >= max_depth:
            return

        for key, value in obj.items():
            describe_checkpoint_object(
                value,
                prefix=str(key),
                depth=depth + 1,
                max_depth=max_depth,
            )

    elif isinstance(obj, torch.Tensor):
        print(
            f"{indent}{prefix}: Tensor "
            f"shape={tuple(obj.shape)}, "
            f"dtype={obj.dtype}, "
            f"device={obj.device}"
        )

    elif isinstance(obj, (list, tuple)):
        print(f"{indent}{prefix}: {type(obj).__name__}, length={len(obj)}")

        if depth < max_depth and len(obj) > 0:
            describe_checkpoint_object(
                obj[0],
                prefix="[0]",
                depth=depth + 1,
                max_depth=max_depth,
            )

    else:
        print(f"{indent}{prefix}: {type(obj).__name__}")


print("\n" + "=" * 75)
print("CHECKPOINT SERIALIZATION INSPECTION")
print("=" * 75)

checkpoint_inspection = {}

for checkpoint_name, checkpoint_path in SAMPLE_CHECKPOINTS.items():

    print(f"\n{checkpoint_name}")
    print("-" * 75)
    print("Path:", checkpoint_path)
    print("Exists:", checkpoint_path.exists())

    if not checkpoint_path.exists():
        continue

    try:
        # Safe loading: do not permit arbitrary Python object
        # deserialization.
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True,
        )

        checkpoint_inspection[checkpoint_name] = checkpoint

        describe_checkpoint_object(checkpoint)

    except Exception as exc:
        print("SAFE LOAD FAILED")
        print("Exception type:", type(exc).__name__)
        print("Exception:", str(exc))


# ------------------------------------------------------------
# 6. Summary and safeguards
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CELL 5 SUMMARY")
print("=" * 75)

print("Checkpoint metadata records:", len(recorded_hashes))
print("Hash verification passed:", all_hashes_match)
print("Representative checkpoints safely loaded:",
      list(checkpoint_inspection.keys()))

print("\nNo model weights were modified.")
print("No training or evaluation was performed.")
print("No validation or test samples were loaded.")
print("No test-set access occurred.")

if not all_hashes_match:
    print("\nWARNING: Resolve checkpoint hash discrepancies before proceeding.")

CHECKPOINT HASH VERIFICATION


strategy,50_50,60_40,65_35,70_30,75_25,80_20,natural
replicate,,,,,,,
1,PASS,PASS,PASS,PASS,PASS,PASS,PASS
2,PASS,PASS,PASS,PASS,PASS,PASS,PASS
3,PASS,PASS,PASS,PASS,PASS,PASS,PASS
4,PASS,PASS,PASS,PASS,PASS,PASS,PASS
5,PASS,PASS,PASS,PASS,PASS,PASS,PASS
6,PASS,PASS,PASS,PASS,PASS,PASS,PASS
7,PASS,PASS,PASS,PASS,PASS,PASS,PASS
8,PASS,PASS,PASS,PASS,PASS,PASS,PASS



Verification summary:
status
PASS    56

All 56 recorded hashes match: True

CHECKPOINT SERIALIZATION INSPECTION

replicate_01_natural
---------------------------------------------------------------------------
Path: /home/ankit/Desktop/qskit/qec_ml/notebooks/qec_model_checkpoints/paired_multiseed_fixed_budget_20261101_154138/checkpoints/replicate_01_natural.pt
Exists: True
root: dict with 5 entries
  trial_key: str
  metadata: dict with 30 entries
    requested_class1_ratio: float
    observed_dataset_class1_ratio: float
    actual_sampled_class1_ratio: float
    sampled_class0_episodes: int
    sampled_class1_episodes: int
    class0_count: int
    class1_count: int
    class0_replacement: bool
    class1_replacement: bool
    trial_seed: int
    dataset_seed: int
    environment_seed: int
    replay_seed: int
    action_seed: int
    warmup_episodes: int
    completed_training_episodes: int
    incomplete_final_episode: bool
    transitions: int
    gradient_updates: int
    target

In [6]:
from pathlib import Path
import json
import re

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")

# Exclude generated data and environments. We only want source code.
EXCLUDED_DIRS = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    "datasets",
    "data",
    "results",
    "checkpoints",
    "qec_model_checkpoints",
    "phase2_freeze_20261104",
    "phase4_final_test_results",
}

SEARCH_PATTERNS = {
    "Q-network definition": re.compile(
        r"class\s+QNetwork\b|class\s+\w*QNetwork\b"
    ),
    "Greedy policy evaluator": re.compile(
        r"def\s+evaluate_greedy_policy\b"
    ),
    "Environment definition": re.compile(
        r"class\s+\w*(?:Env|Environment)\b"
    ),
    "Observation construction": re.compile(
        r"observation_space|observation\s*=|def\s+_get_obs|def\s+get_obs"
    ),
    "Action mapping": re.compile(
        r"action_to|action_map|STOP_ACTION|stop_action|n_actions|action_dim"
    ),
    "Terminal correction logic": re.compile(
        r"D_residual|L_residual|residual_detectors|logical_observable|"
        r"correction_vector|correction_vectors|terminal.*correction"
    ),
    "Evaluation function calls": re.compile(
        r"evaluate_greedy_policy\s*\("
    ),
}

def is_excluded(path):
    return any(part in EXCLUDED_DIRS for part in path.parts)

def extract_code_sources(root):
    sources = []

    for path in root.rglob("*.py"):
        if is_excluded(path):
            continue
        try:
            sources.append((path, path.read_text(encoding="utf-8")))
        except Exception as exc:
            print(f"Could not read {path}: {exc}")

    for path in root.rglob("*.ipynb"):
        if is_excluded(path):
            continue
        try:
            notebook = json.loads(path.read_text(encoding="utf-8"))
            for cell_index, cell in enumerate(notebook.get("cells", [])):
                if cell.get("cell_type") != "code":
                    continue

                source = cell.get("source", "")
                if isinstance(source, list):
                    source = "".join(source)

                if source.strip():
                    label = Path(
                        f"{path} [code cell {cell_index}]"
                    )
                    sources.append((label, source))

        except Exception as exc:
            print(f"Could not inspect notebook {path}: {exc}")

    return sources


sources = extract_code_sources(PROJECT_ROOT)

print("=" * 100)
print("SOURCE INVENTORY")
print("=" * 100)
print(f"Project root: {PROJECT_ROOT}")
print(f"Python/notebook code sources inspected: {len(sources)}")

for category, pattern in SEARCH_PATTERNS.items():
    matches = []

    for path, source in sources:
        lines = source.splitlines()

        for line_number, line in enumerate(lines, start=1):
            if pattern.search(line):
                matches.append((path, line_number, lines))

    print("\n" + "=" * 100)
    print(category.upper())
    print("=" * 100)
    print(f"Matching locations: {len(matches)}")

    if not matches:
        print("No matches found.")
        continue

    # Show a limited number of relevant excerpts per category.
    for path, line_number, lines in matches[:8]:
        start = max(0, line_number - 7)
        end = min(len(lines), line_number + 12)

        print(f"\n--- {path} | line {line_number} ---")
        for index in range(start, end):
            print(f"{index + 1:5}: {lines[index]}")

    if len(matches) > 8:
        print(f"\n... {len(matches) - 8} additional matching locations omitted.")

print("\n" + "=" * 100)
print("READ-ONLY SAFETY SUMMARY")
print("=" * 100)
print("Only .py source files and .ipynb code cells were inspected.")
print("No Python project modules were imported.")
print("No checkpoints or model weights were loaded.")
print("No training or evaluation was performed.")
print("No dataset arrays, validation samples, or test samples were loaded.")
print("No files were modified.")

SOURCE INVENTORY
Project root: /home/ankit/Desktop/qskit/qec_ml
Python/notebook code sources inspected: 738

Q-NETWORK DEFINITION
Matching locations: 4

--- /home/ankit/Desktop/qskit/qec_ml/notebooks/RL_decoder_vectorized.ipynb [code cell 13] | line 44 ---
   38: 
   39: 
   40: # ------------------------------------------------------------
   41: # 2. Q-network
   42: # ------------------------------------------------------------
   43: 
   44: class QNetwork(nn.Module):
   45:     """
   46:     Estimates Q(s, a) for every available action.
   47: 
   48:     Input:
   49:         State vector containing detector bits and correction frame.
   50: 
   51:     Output:
   52:         One Q-value for each action, including STOP.
   53:     """
   54: 
   55:     def __init__(self, obs_dim, action_dim, hidden_dim=128):
   56:         super().__init__()

--- /home/ankit/Desktop/qskit/qec_ml/notebooks/RL_decoder_v2.ipynb [code cell 16] | line 43 ---
   37:     print("GPU:", torch.cuda.get_d

In [7]:
from pathlib import Path
import json
import re

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "RL_decoder_v2.ipynb"

# ------------------------------------------------------------
# 1. Read the relevant notebook cells as source text
# ------------------------------------------------------------

notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
cells = notebook.get("cells", [])

CELL_INDICES = [15, 16, 17, 35, 36, 40, 46]

print("=" * 100)
print("RL_decoder_v2.ipynb — SELECTED SOURCE CELLS")
print("=" * 100)

for cell_index in CELL_INDICES:
    if cell_index >= len(cells):
        print(f"\nCell {cell_index}: DOES NOT EXIST")
        continue

    cell = cells[cell_index]
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    print("\n" + "=" * 100)
    print(f"NOTEBOOK CELL {cell_index} | type={cell.get('cell_type')}")
    print("=" * 100)

    if cell.get("cell_type") != "code":
        print(source[:3000])
        continue

    print(source if source.strip() else "[EMPTY CODE CELL]")


# ------------------------------------------------------------
# 2. Search for the exact paired-experiment/checkpoint code
# ------------------------------------------------------------

EXCLUDED_DIRS = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    "datasets",
    "data",
    "results",
    "checkpoints",
    "qec_model_checkpoints",
    "phase2_freeze_20261104",
    "phase4_final_test_results",
}

SEARCH_TERMS = {
    "Paired experiment identifier": re.compile(
        r"paired_multiseed_fixed_budget"
    ),
    "Checkpoint filename construction": re.compile(
        r"replicate_\{.*\}.*\.pt|replicate_.*\.pt"
    ),
    "Checkpoint save call": re.compile(
        r"torch\.save\s*\("
    ),
    "Online network checkpoint key": re.compile(
        r"online_network_state_dict"
    ),
    "Recorded checkpoint hash": re.compile(
        r"checkpoint_sha256|initialization_sha256"
    ),
    "Validation archive identifier": re.compile(
        r"validation_evaluation_56_models"
    ),
}

def excluded(path):
    return any(part in EXCLUDED_DIRS for part in path.parts)

source_files = []

for path in PROJECT_ROOT.rglob("*.py"):
    if not excluded(path):
        try:
            source_files.append((path, path.read_text(encoding="utf-8")))
        except Exception:
            pass

for path in PROJECT_ROOT.rglob("*.ipynb"):
    if excluded(path):
        continue

    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
        for index, cell in enumerate(nb.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue

            source = cell.get("source", "")
            if isinstance(source, list):
                source = "".join(source)

            if source.strip():
                source_files.append(
                    (Path(f"{path} [cell {index}]"), source)
                )
    except Exception:
        pass


print("\n" + "=" * 100)
print("PAIRED EXPERIMENT AND CHECKPOINT SOURCE SEARCH")
print("=" * 100)

for label, pattern in SEARCH_TERMS.items():
    print("\n" + "-" * 100)
    print(label.upper())
    print("-" * 100)

    found = 0

    for path, source in source_files:
        lines = source.splitlines()

        matching_lines = [
            i for i, line in enumerate(lines)
            if pattern.search(line)
        ]

        if not matching_lines:
            continue

        found += len(matching_lines)

        # Show at most three excerpts per file for this search term.
        for line_index in matching_lines[:3]:
            start = max(0, line_index - 8)
            end = min(len(lines), line_index + 13)

            print(f"\n--- {path} | line {line_index + 1} ---")
            for i in range(start, end):
                print(f"{i + 1:5}: {lines[i]}")

    print(f"\nTotal matching lines: {found}")


# ------------------------------------------------------------
# 3. Safety summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SAFETY SUMMARY")
print("=" * 100)
print("Notebook cells were read as source text only.")
print("Project source files were searched as text only.")
print("No notebook cells or project modules were executed.")
print("No checkpoints or model weights were loaded.")
print("No training or evaluation was performed.")
print("No dataset arrays or test samples were accessed.")
print("No files were modified.")

RL_decoder_v2.ipynb — SELECTED SOURCE CELLS

NOTEBOOK CELL 15 | type=code

# ============================================================
# CELL 13 — QUANTUM FEEDBACK ENVIRONMENT V2
# No repeated correction actions
# ============================================================

import numpy as np


class QuantumFeedbackEnv:

    def __init__(
        self,
        D_history,
        D_final,
        L_target,
        H_terminal,
        logical_effect,
        data_qubits,
        seed=42,
    ):

        self.D_history = np.asarray(D_history, dtype=np.uint8)
        self.D_final = np.asarray(D_final, dtype=np.uint8)
        self.L_target = np.asarray(L_target, dtype=np.uint8)

        self.H_terminal = np.asarray(
            H_terminal, dtype=np.uint8
        )

        self.logical_effect = np.asarray(
            logical_effect, dtype=np.uint8
        )

        self.data_qubits = np.asarray(data_qubits)

        self.num_samples = len(self.L_target)
        self.num_correction_act

In [8]:
# =============================================================================
# PHASE 4 — CELL 8
# READ-ONLY SOURCE INSPECTION:
# TRAINING, ACTION SELECTION, AND DATA-SPLIT PROVENANCE
#
# Safety:
#   - Does not execute source notebook cells.
#   - Does not load datasets or NumPy arrays.
#   - Does not load model checkpoints.
#   - Does not access test samples or test labels.
#   - Does not train or evaluate any model.
#   - Does not modify any files.
# =============================================================================

import json
import re
from pathlib import Path

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "RL_decoder_v2.ipynb"

assert NOTEBOOK_PATH.is_file(), f"Notebook not found: {NOTEBOOK_PATH}"

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

def source_text(cell):
    source = cell.get("source", "")
    if isinstance(source, list):
        return "".join(source)
    return str(source)

print("=" * 100)
print("PHASE 4 — CELL 8: SOURCE PROVENANCE INSPECTION")
print("=" * 100)
print("Notebook:", NOTEBOOK_PATH)
print("Notebook cells:", len(cells))
print("Mode: READ-ONLY SOURCE INSPECTION")
print("Dataset arrays loaded: NO")
print("Checkpoint files loaded: NO")
print("Test samples accessed: NO")


# -----------------------------------------------------------------------------
# 1. Inspect the exact training and validation source cells.
# -----------------------------------------------------------------------------

selected_cell_ids = [109, 112, 118, 121, 123, 124, 129, 130, 131, 132]

print("\n" + "=" * 100)
print("SECTION 1 — SELECTED EXPERIMENT / EVALUATION CELLS")
print("=" * 100)

for cell_id in selected_cell_ids:
    if cell_id >= len(cells):
        continue

    cell = cells[cell_id]
    source = source_text(cell)

    print("\n" + "-" * 100)
    print(f"CELL {cell_id} | type={cell.get('cell_type')}")
    print("-" * 100)

    if not source.strip():
        print("[EMPTY CELL]")
        continue

    # Keep output manageable while retaining complete shorter cells.
    max_chars = 18000
    if len(source) > max_chars:
        print(source[:max_chars])
        print(
            f"\n[TRUNCATED: {len(source) - max_chars} additional characters]"
        )
    else:
        print(source)


# -----------------------------------------------------------------------------
# 2. Locate action-selection definitions and their dependencies.
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 2 — ACTION SELECTION DEFINITIONS")
print("=" * 100)

action_patterns = [
    re.compile(r"^\s*def\s+select_action\s*\(", re.MULTILINE),
    re.compile(r"^\s*def\s+evaluate_greedy_policy\s*\(", re.MULTILINE),
]

action_matches = []

for i, cell in enumerate(cells):
    source = source_text(cell)

    if any(pattern.search(source) for pattern in action_patterns):
        action_matches.append((i, source))

for cell_id, source in action_matches:
    print("\n" + "-" * 100)
    print(f"CELL {cell_id}")
    print("-" * 100)
    print(source[:14000])

if not action_matches:
    print("No matching function definitions found.")


# -----------------------------------------------------------------------------
# 3. Locate original dataset generation and split definitions.
#    This searches source text only; it does not load any arrays.
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 3 — DATA GENERATION AND SPLIT SOURCE")
print("=" * 100)

split_patterns = [
    re.compile(
        r"train_test_split|random_split|StratifiedShuffleSplit",
        re.IGNORECASE,
    ),
    re.compile(
        r"train_indices|val_indices|validation_indices|test_indices",
        re.IGNORECASE,
    ),
    re.compile(
        r"TRAIN_SIZE|VALIDATION_SIZE|TEST_SIZE|N_TRAIN|N_VALIDATION|N_TEST",
        re.IGNORECASE,
    ),
    re.compile(
        r"random_state|split_seed|SPLIT_SEED",
        re.IGNORECASE,
    ),
    re.compile(
        r"np\.savez|np\.save\(|savez_compressed|\.npz",
        re.IGNORECASE,
    ),
    re.compile(
        r"train_D_history|val_D_history|test_D_history",
        re.IGNORECASE,
    ),
    re.compile(
        r"train_L_target|val_L_target|test_L_target",
        re.IGNORECASE,
    ),
]

split_matches = []

for i, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    source = source_text(cell)
    matching_lines = []

    lines = source.splitlines()

    for line_number, line in enumerate(lines, start=1):
        if any(pattern.search(line) for pattern in split_patterns):
            matching_lines.append((line_number, line))

    if matching_lines:
        split_matches.append((i, source, matching_lines))

for cell_id, source, matching_lines in split_matches:
    print("\n" + "-" * 100)
    print(f"CELL {cell_id} — {len(matching_lines)} matching lines")
    print("-" * 100)

    lines = source.splitlines()
    selected_line_numbers = set()

    # Include surrounding context for each matching line.
    for line_number, _ in matching_lines:
        for j in range(max(1, line_number - 5),
                       min(len(lines), line_number + 5) + 1):
            selected_line_numbers.add(j)

    previous = None

    for line_number in sorted(selected_line_numbers):
        if previous is not None and line_number > previous + 1:
            print("    ...")

        print(f"{line_number:5d}: {lines[line_number - 1]}")
        previous = line_number


# -----------------------------------------------------------------------------
# 4. Inventory dataset-like files by filename and metadata only.
#    Do not open .npz, .npy, .pt, or other data contents.
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 4 — DATASET FILE INVENTORY (NAMES AND FILE METADATA ONLY)")
print("=" * 100)

search_roots = [
    PROJECT_ROOT,
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
]

seen = set()

relevant_extensions = {
    ".npz", ".npy", ".csv", ".json", ".pkl", ".pickle",
    ".pt", ".pth", ".h5", ".hdf5",
}

name_pattern = re.compile(
    r"dataset|data|split|index|indices|train|valid|test|surface|syndrome",
    re.IGNORECASE,
)

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if ".ipynb_checkpoints" in path.parts:
            continue

        if path.suffix.lower() not in relevant_extensions:
            continue

        if not name_pattern.search(path.name):
            continue

        resolved = str(path.resolve())

        if resolved in seen:
            continue

        seen.add(resolved)

        try:
            stat = path.stat()
            print(
                f"{path}\n"
                f"    size_bytes={stat.st_size:,} "
                f"modified_ns={stat.st_mtime_ns}"
            )
        except OSError as exc:
            print(f"{path}\n    metadata_error={exc}")


# -----------------------------------------------------------------------------
# 5. Safety summary.
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 8 SAFETY SUMMARY")
print("=" * 100)
print("Notebook source read as text: YES")
print("Project source executed: NO")
print("Dataset files opened: NO")
print("NumPy arrays loaded: NO")
print("Test samples inspected: NO")
print("Checkpoints loaded: NO")
print("Training performed: NO")
print("Evaluation performed: NO")
print("Files modified: NO")

PHASE 4 — CELL 8: SOURCE PROVENANCE INSPECTION
Notebook: /home/ankit/Desktop/qskit/qec_ml/notebooks/RL_decoder_v2.ipynb
Notebook cells: 160
Mode: READ-ONLY SOURCE INSPECTION
Dataset arrays loaded: NO
Checkpoint files loaded: NO
Test samples accessed: NO

SECTION 1 — SELECTED EXPERIMENT / EVALUATION CELLS

----------------------------------------------------------------------------------------------------
CELL 109 | type=code
----------------------------------------------------------------------------------------------------
# =============================================================================
# READ-ONLY MULTI-SEED CHECKPOINT INVENTORY AND STRUCTURAL AUDIT
# =============================================================================

import os
import json
import hashlib
from pathlib import Path
from collections import Counter

import torch
import numpy as np

print("=" * 88)
print("MULTI-SEED CHECKPOINT INVENTORY AND STRUCTURAL AUDIT")
print("=" * 88)

# -------------------

In [10]:

from pathlib import Path
import json
import hashlib
import zipfile
import numpy as np

# ============================================================
# PHASE 4 — CELL 9 (CORRECTED)
# Dataset and split metadata audit
#
# Read-only:
#   - Reads JSON metadata
#   - Computes file hashes
#   - Reads NPZ/NPY headers only
#
# Does NOT:
#   - Load dataset arrays
#   - Read train/validation/test index values
#   - Inspect test samples
#   - Train or evaluate models
# ============================================================

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")
DATASET_DIR = PROJECT_ROOT / "datasets"

DATASET_FILE = DATASET_DIR / "surface_code_d3_r3_p001.npz"
SPLIT_FILE = DATASET_DIR / "surface_code_d3_r3_p001_splits.npz"

DATASET_METADATA_FILE = (
    DATASET_DIR / "surface_code_d3_r3_p001_metadata.json"
)
SPLIT_METADATA_FILE = (
    DATASET_DIR / "surface_code_d3_r3_p001_splits_metadata.json"
)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def inspect_npz_headers(path):
    """
    Inspect member names and NPY headers inside an NPZ archive.
    Does not decompress/load array payloads.
    """
    result = {}

    with zipfile.ZipFile(path, "r") as archive:
        for member in archive.namelist():
            if not member.endswith(".npy"):
                continue

            array_name = Path(member).stem

            with archive.open(member, "r") as f:
                version = np.lib.format.read_magic(f)

                if version == (1, 0):
                    shape, fortran_order, dtype = (
                        np.lib.format.read_array_header_1_0(f)
                    )
                elif version == (2, 0):
                    shape, fortran_order, dtype = (
                        np.lib.format.read_array_header_2_0(f)
                    )
                elif version == (3, 0):
                    shape, fortran_order, dtype = (
                        np.lib.format.read_array_header_2_0(f)
                    )
                else:
                    raise ValueError(
                        f"Unsupported NPY format {version} in {member}"
                    )

            result[array_name] = {
                "shape": shape,
                "dtype": str(dtype),
                "fortran_order": fortran_order,
            }

    return result


# ------------------------------------------------------------
# 1. Confirm required files
# ------------------------------------------------------------

required_files = [
    DATASET_FILE,
    SPLIT_FILE,
    DATASET_METADATA_FILE,
    SPLIT_METADATA_FILE,
]

for path in required_files:
    assert path.is_file(), f"Missing required file: {path}"

print("=" * 88)
print("PHASE 4 — CELL 9: DATASET AND SPLIT METADATA AUDIT")
print("=" * 88)


# ------------------------------------------------------------
# 2. Read metadata only
# ------------------------------------------------------------

with open(DATASET_METADATA_FILE, "r") as f:
    dataset_metadata = json.load(f)

with open(SPLIT_METADATA_FILE, "r") as f:
    split_metadata = json.load(f)

print("\n[1] DATASET METADATA")
print("-" * 88)
print(json.dumps(dataset_metadata, indent=2))

print("\n[2] SPLIT METADATA")
print("-" * 88)
print(json.dumps(split_metadata, indent=2))


# ------------------------------------------------------------
# 3. Verify metadata consistency
# ------------------------------------------------------------

assert dataset_metadata["experiment"] == "rotated_memory_x"
assert dataset_metadata["distance"] == 3
assert dataset_metadata["rounds"] == 3
assert dataset_metadata["noise_rate"] == 0.01
assert dataset_metadata["shots"] == 20000

assert split_metadata["dataset_file"] == DATASET_FILE.name
assert split_metadata["split_file"] == SPLIT_FILE.name
assert split_metadata["random_seed"] == 123
assert split_metadata["stratification_variable"] == "L_target"

assert split_metadata["train_ratio"] == 0.70
assert split_metadata["validation_ratio"] == 0.15
assert split_metadata["test_ratio"] == 0.15

assert split_metadata["total_samples"] == 20000
assert split_metadata["train_samples"] == 13999
assert split_metadata["validation_samples"] == 2999
assert split_metadata["test_samples"] == 3002

partition_sum = (
    split_metadata["train_samples"]
    + split_metadata["validation_samples"]
    + split_metadata["test_samples"]
)

assert partition_sum == split_metadata["total_samples"]

print("\n[3] RECORDED SPLIT CONFIGURATION")
print("-" * 88)
print(f"Dataset-generation seed: {dataset_metadata['random_seed']}")
print(f"Split-generation seed:   {split_metadata['random_seed']}")
print(f"Stratification:          {split_metadata['stratification_variable']}")
print(f"Train samples:           {split_metadata['train_samples']}")
print(f"Validation samples:      {split_metadata['validation_samples']}")
print(f"Test samples:            {split_metadata['test_samples']}")
print(f"Partition counts sum to total: {partition_sum == 20000}")


# ------------------------------------------------------------
# 4. File sizes and SHA256 hashes
# ------------------------------------------------------------

print("\n[4] DATASET FILE INTEGRITY")
print("-" * 88)

for path in [DATASET_FILE, SPLIT_FILE]:
    print(f"\nFile:   {path.name}")
    print(f"Size:   {path.stat().st_size:,} bytes")
    print(f"SHA256: {sha256_file(path)}")


# ------------------------------------------------------------
# 5. Inspect dataset archive headers
# ------------------------------------------------------------

dataset_headers = inspect_npz_headers(DATASET_FILE)
split_headers = inspect_npz_headers(SPLIT_FILE)

print("\n[5] ORIGINAL DATASET ARCHIVE HEADERS")
print("-" * 88)

for name, info in sorted(dataset_headers.items()):
    print(
        f"{name:28s} "
        f"shape={str(info['shape']):22s} "
        f"dtype={info['dtype']:10s} "
        f"Fortran={info['fortran_order']}"
    )

print("\n[6] SPLIT ARCHIVE HEADERS")
print("-" * 88)

for name, info in sorted(split_headers.items()):
    print(
        f"{name:28s} "
        f"shape={str(info['shape']):22s} "
        f"dtype={info['dtype']:10s} "
        f"Fortran={info['fortran_order']}"
    )


# ------------------------------------------------------------
# 6. Validate the observed dataset schema
# ------------------------------------------------------------

expected_dataset_shapes = {
    "D_history": (20000, 20),
    "D_final": (20000, 4),
    "L_target": (20000,),
    "all_detectors": (20000, 24),
    "all_observables": (20000, 1),
    "data_qubits": (9,),
    "terminal_detector_effects": (9, 24),
    "terminal_observable_effects": (9, 1),
}

expected_split_shapes = {
    "train_indices": (13999,),
    "validation_indices": (2999,),
    "test_indices": (3002,),
}

assert set(dataset_headers) == set(expected_dataset_shapes), (
    "Unexpected dataset archive schema.\n"
    f"Expected: {sorted(expected_dataset_shapes)}\n"
    f"Found:    {sorted(dataset_headers)}"
)

assert set(split_headers) == set(expected_split_shapes), (
    "Unexpected split archive schema.\n"
    f"Expected: {sorted(expected_split_shapes)}\n"
    f"Found:    {sorted(split_headers)}"
)

for name, expected_shape in expected_dataset_shapes.items():
    actual_shape = dataset_headers[name]["shape"]
    assert actual_shape == expected_shape, (
        f"{name}: expected {expected_shape}, got {actual_shape}"
    )

for name, expected_shape in expected_split_shapes.items():
    actual_shape = split_headers[name]["shape"]
    assert actual_shape == expected_shape, (
        f"{name}: expected {expected_shape}, got {actual_shape}"
    )


# ------------------------------------------------------------
# 7. Validate key data types from the observed archive
# ------------------------------------------------------------

for name in [
    "D_history",
    "D_final",
    "L_target",
    "all_detectors",
    "all_observables",
    "terminal_detector_effects",
    "terminal_observable_effects",
]:
    assert dataset_headers[name]["dtype"] == "uint8", (
        f"Unexpected dtype for {name}: "
        f"{dataset_headers[name]['dtype']}"
    )

assert dataset_headers["data_qubits"]["dtype"] == "int64"

for name in expected_split_shapes:
    assert split_headers[name]["dtype"] == "int64"


# ------------------------------------------------------------
# 8. Final audit summary
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 9 AUDIT SUMMARY")
print("=" * 88)

print("[PASS] Dataset metadata is internally consistent.")
print("[PASS] Split metadata is internally consistent.")
print("[PASS] Dataset archive contains all eight expected arrays.")
print("[PASS] Split archive contains all three expected index arrays.")
print("[PASS] All observed array shapes and dtypes match.")
print("[PASS] Dataset and split file hashes recorded.")

print("\nSAFETY CHECKS")
print("-" * 88)
print("[PASS] No dataset array payloads loaded.")
print("[PASS] No train/validation/test index values read.")
print("[PASS] No test samples inspected.")
print("[PASS] No model loaded.")
print("[PASS] No training or evaluation performed.")

PHASE 4 — CELL 9: DATASET AND SPLIT METADATA AUDIT

[1] DATASET METADATA
----------------------------------------------------------------------------------------
{
  "experiment": "rotated_memory_x",
  "distance": 3,
  "rounds": 3,
  "noise_rate": 0.01,
  "shots": 20000,
  "random_seed": 42,
  "data_qubits": [
    1,
    3,
    5,
    8,
    10,
    12,
    15,
    17,
    19
  ],
  "num_detectors": 24,
  "num_observables": 1,
  "historical_detector_count": 20,
  "final_detector_count": 4,
  "logical_observable_count": 1,
  "created_utc": "2026-10-30T10:07:03.692329+00:00",
  "dataset_file": "surface_code_d3_r3_p001.npz"
}

[2] SPLIT METADATA
----------------------------------------------------------------------------------------
{
  "dataset_file": "surface_code_d3_r3_p001.npz",
  "split_file": "surface_code_d3_r3_p001_splits.npz",
  "random_seed": 123,
  "stratification_variable": "L_target",
  "train_ratio": 0.7,
  "validation_ratio": 0.15,
  "test_ratio": 0.15,
  "total_samples": 2

In [11]:

# ============================================================
# PHASE 4 — CELL 10
# Read-only inspection of the original QEC/RL implementation
#
# Purpose:
#   1. Inspect the environment and observation construction.
#   2. Inspect the Q-network architecture.
#   3. Compare the original and later evaluation functions.
#   4. Inspect training and validation checkpoint-loading logic.
#   5. Establish which implementation should be used for
#      final held-out test evaluation.
#
# SAFETY:
#   - Reads notebook JSON/source text only.
#   - Does not execute any notebook cell.
#   - Does not import project modules.
#   - Does not load dataset arrays or split indices.
#   - Does not load checkpoints.
#   - Does not access test samples.
# ============================================================

from pathlib import Path
import json

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "RL_decoder_v2.ipynb"

assert NOTEBOOK_PATH.is_file(), (
    f"Original notebook not found: {NOTEBOOK_PATH}"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

print("=" * 100)
print("PHASE 4 — CELL 10: ORIGINAL IMPLEMENTATION SOURCE AUDIT")
print("=" * 100)

print(f"\nNotebook: {NOTEBOOK_PATH}")
print(f"Total cells: {len(cells)}")

# These are the source cells identified during the previous
# source-only audit. We inspect their text, not their execution.
TARGET_CELLS = [
    8,       # Dataset generation
    9,       # Original train/validation/test split
    10,      # Dataset loading and training-data extraction
    15,      # Quantum feedback environment
    16,      # QNetwork definition
    35,      # Reward-shaped environment variant
    36,      # Related reward/environment implementation
    40,      # Earlier greedy evaluator
    46,      # Later evaluator with correction-pattern recording
    102,     # Paired multi-seed experiment setup
    104,     # Paired experiment training implementation
    105,     # Related training/checkpoint implementation
    112,     # Validation checkpoint-loading/evaluation
    118,     # Extension training implementation
]

for cell_index in TARGET_CELLS:
    print("\n" + "=" * 100)
    print(f"SOURCE CELL {cell_index}")
    print("=" * 100)

    if cell_index >= len(cells):
        print("[NOT FOUND] Notebook does not contain this cell index.")
        continue

    cell = cells[cell_index]
    cell_type = cell.get("cell_type", "unknown")
    source = cell.get("source", "")

    if isinstance(source, list):
        source = "".join(source)

    print(f"Cell type: {cell_type}")
    print(f"Source length: {len(source):,} characters")
    print("-" * 100)

    if not source.strip():
        print("[EMPTY CELL]")
    else:
        print(source)

print("\n" + "=" * 100)
print("CELL 10 SAFETY SUMMARY")
print("=" * 100)
print("[PASS] Original notebook opened as JSON.")
print("[PASS] Only source text from selected cells was inspected.")
print("[PASS] No notebook cell was executed.")
print("[PASS] No project module was imported.")
print("[PASS] No dataset arrays or split indices were loaded.")
print("[PASS] No checkpoint was loaded.")
print("[PASS] No test sample was accessed.")

print("\nCell 10 completed. Review the source output before proceeding.")


PHASE 4 — CELL 10: ORIGINAL IMPLEMENTATION SOURCE AUDIT

Notebook: /home/ankit/Desktop/qskit/qec_ml/notebooks/RL_decoder_v2.ipynb
Total cells: 160

SOURCE CELL 8
Cell type: code
Source length: 4,176 characters
----------------------------------------------------------------------------------------------------

# ============================================================
# CELL 6 — GENERATE AND SAVE CIRCUIT-LEVEL RL DATASET
# ============================================================

import numpy as np
import json
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DATASET_SHOTS = 20000
RANDOM_SEED = 42

# Save inside the project directory
PROJECT_DIR = Path("/home/ankit/Desktop/qskit/qec_ml")
DATASET_DIR = PROJECT_DIR / "datasets"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATASET_DIR / "surface_code_d3

In [12]:

# ============================================================
# PHASE 4 — CELL 11
# Locate missing RL helper definitions and the 56-model
# validation evaluator.
#
# READ-ONLY SOURCE INSPECTION ONLY.
# ============================================================

from pathlib import Path
import json
import re

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "RL_decoder_v2.ipynb"

assert NOTEBOOK_PATH.is_file()

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

def source_text(cell):
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source

# ------------------------------------------------------------
# 1. Locate exact helper definitions
# ------------------------------------------------------------

helper_patterns = {
    "select_action": re.compile(
        r"^\s*def\s+select_action\s*\(", re.MULTILINE
    ),
    "run_fixed_budget_trial": re.compile(
        r"^\s*def\s+run_fixed_budget_trial\s*\(", re.MULTILINE
    ),
}

print("=" * 100)
print("PHASE 4 — CELL 11: HELPER AND EVALUATOR SOURCE DISCOVERY")
print("=" * 100)

helper_cells = {}

for name, pattern in helper_patterns.items():
    helper_cells[name] = []

    for index, cell in enumerate(cells):
        source = source_text(cell)

        if pattern.search(source):
            helper_cells[name].append(index)

    print(f"\n{name}:")
    print("Definition cells:", helper_cells[name])

# Print complete source for the helper definitions.
for name, indices in helper_cells.items():
    for index in indices:
        print("\n" + "=" * 100)
        print(f"HELPER DEFINITION: {name} — CELL {index}")
        print("=" * 100)
        print(source_text(cells[index]))

# ------------------------------------------------------------
# 2. Locate validation evaluation cells for 56 models
# ------------------------------------------------------------

candidate_patterns = {
    "56-model count": re.compile(
        r"number_of_models.{0,20}56|models.{0,20}56",
        re.IGNORECASE,
    ),
    "8-replicate schedule": re.compile(
        r"REPLICATES\s*=\s*\[[^\]]*8[^\]]*\]",
        re.IGNORECASE,
    ),
    "56-model validation archive": re.compile(
        r"validation_evaluation_56_models",
        re.IGNORECASE,
    ),
    "all 56 expected trials": re.compile(
        r"EXPECTED_TRIAL_KEYS|EXPECTED_TRIALS",
        re.IGNORECASE,
    ),
}

candidate_cells = {}

for label, pattern in candidate_patterns.items():
    matches = []

    for index, cell in enumerate(cells):
        source = source_text(cell)

        if pattern.search(source):
            matches.append(index)

    candidate_cells[label] = matches

    print("\n" + "-" * 100)
    print(label)
    print("Matching cells:", matches)

# ------------------------------------------------------------
# 3. Print likely 56-model validation cells
# ------------------------------------------------------------

# Exclude cells already inspected in Cell 10.
already_inspected = {
    8, 9, 10, 15, 16, 35, 36, 40, 46, 102, 104, 105, 112, 118
}

likely_validation_cells = set()

for label, indices in candidate_cells.items():
    if label in {
        "56-model count",
        "8-replicate schedule",
        "56-model validation archive",
    }:
        likely_validation_cells.update(indices)

likely_validation_cells -= already_inspected

# Only print cells whose source looks like an evaluation cell.
evaluation_terms = (
    "evaluate_greedy_policy",
    "evaluation_metadata",
    "validation_evaluation",
    "per_sample_predictions",
    "greedy_validation",
    "validation_samples",
)

for index in sorted(likely_validation_cells):
    source = source_text(cells[index])

    if any(term in source for term in evaluation_terms):
        print("\n" + "=" * 100)
        print(f"POTENTIAL 56-MODEL VALIDATION CELL {index}")
        print("=" * 100)
        print(source)

# ------------------------------------------------------------
# 4. Safety summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 11 SAFETY SUMMARY")
print("=" * 100)
print("[PASS] Notebook source read as JSON.")
print("[PASS] No notebook cells executed.")
print("[PASS] No project modules imported.")
print("[PASS] No dataset arrays or split indices loaded.")
print("[PASS] No checkpoints loaded.")
print("[PASS] No test samples accessed.")
print("[PASS] No project files modified.")

print("\nCell 11 completed. Review the source output.")


PHASE 4 — CELL 11: HELPER AND EVALUATOR SOURCE DISCOVERY

select_action:
Definition cells: [19]

run_fixed_budget_trial:
Definition cells: [85]

HELPER DEFINITION: select_action — CELL 19

# ============================================================
# CELL 17 — EPSILON-GREEDY POLICY AND EPISODE GENERATION
# ============================================================

import numpy as np
import torch


# ============================================================
# HYPERPARAMETERS
# ============================================================

INITIAL_EPSILON = 1.0

WARMUP_EPISODES = 100

MAX_EPISODE_STEPS = 10


# ============================================================
# EPSILON-GREEDY ACTION SELECTION
# ============================================================

def select_action(
    observation,
    valid_action_mask,
    epsilon,
    network=online_network,
):

    """
    Select an action using epsilon-greedy exploration.

    Invalid actions are never selected.

    eps

In [13]:
# ============================================================
# PHASE 4 — CELL 12
# Audit missing RL training helpers and the original
# 56-policy validation evaluation implementation.
#
# READ-ONLY SOURCE INSPECTION ONLY.
# ============================================================

from pathlib import Path
import json
import re

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "RL_decoder_v2.ipynb"

assert NOTEBOOK_PATH.is_file(), (
    f"Original notebook not found: {NOTEBOOK_PATH}"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

def get_source(cell):
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source


sources = [get_source(cell) for cell in cells]

print("=" * 110)
print("PHASE 4 — CELL 12: TRAINING HELPERS AND VALIDATION EVALUATOR AUDIT")
print("=" * 110)
print("Notebook:", NOTEBOOK_PATH)
print("Total notebook cells:", len(cells))


# ------------------------------------------------------------
# 1. Locate missing helper definitions
# ------------------------------------------------------------

helper_names = [
    "select_experiment_action",
    "double_dqn_update",
    "make_ratio_dataset",
    "ReplayBuffer",
]

definition_patterns = {
    name: re.compile(
        rf"^\s*(?:def|class)\s+{re.escape(name)}\s*[\(:]",
        re.MULTILINE,
    )
    for name in helper_names
}

helper_matches = {}

for name, pattern in definition_patterns.items():
    matches = []

    for index, source in enumerate(sources):
        if pattern.search(source):
            matches.append(index)

    helper_matches[name] = matches

    print("\n" + "-" * 100)
    print(f"DEFINITION SEARCH: {name}")
    print("Matching cells:", matches)


# ------------------------------------------------------------
# 2. Print complete helper-definition cells
# ------------------------------------------------------------

printed_helper_cells = set()

for name, indices in helper_matches.items():
    for index in indices:
        if index in printed_helper_cells:
            continue

        printed_helper_cells.add(index)

        print("\n" + "=" * 110)
        print(f"COMPLETE HELPER SOURCE — CELL {index}")
        print("=" * 110)
        print(sources[index])


# ------------------------------------------------------------
# 3. Locate action-selection and Double-DQN call sites
# ------------------------------------------------------------

call_patterns = {
    "select_experiment_action calls": re.compile(
        r"\bselect_experiment_action\s*\("
    ),
    "double_dqn_update calls": re.compile(
        r"\bdouble_dqn_update\s*\("
    ),
    "make_ratio_dataset calls": re.compile(
        r"\bmake_ratio_dataset\s*\("
    ),
}

for label, pattern in call_patterns.items():
    matches = [
        index
        for index, source in enumerate(sources)
        if pattern.search(source)
    ]

    print("\n" + "-" * 100)
    print(label)
    print("Matching cells:", matches)


# ------------------------------------------------------------
# 4. Find the actual 56-policy validation evaluation loop
# ------------------------------------------------------------

evaluation_terms = [
    "validation_evaluation_56_models",
    "per_sample_predictions.npz",
    "overall_validation_metrics.csv",
    "completed_trials",
    "evaluate_greedy_policy",
    "checkpoint_path",
    "N_REPLICATES",
    "STRATEGIES",
]

evaluation_candidates = {}

for term in evaluation_terms:
    matches = [
        index
        for index, source in enumerate(sources)
        if term.lower() in source.lower()
    ]

    evaluation_candidates[term] = matches

    print("\n" + "-" * 100)
    print(f"EVALUATION SEARCH: {term}")
    print("Matching cells:", matches)


# ------------------------------------------------------------
# 5. Print likely original evaluation cells
# ------------------------------------------------------------

# These are source-inspection candidates, not assumed evaluators.
# Include cells that mention the archive, evaluator, checkpoint
# loading, or validation progress.

candidate_indices = set()

for term, indices in evaluation_candidates.items():
    candidate_indices.update(indices)

# Exclude analysis-only cells already identified in Cell 11.
# We still include archive creation / evaluation cells.
analysis_only_cells = {
    131, 132, 133, 134, 135, 136, 137
}

candidate_indices -= analysis_only_cells

# Only print cells containing multiple relevant evaluation indicators.
evaluation_indicators = [
    "evaluate_greedy_policy",
    "validation_evaluation_56_models",
    "per_sample_predictions",
    "overall_validation_metrics",
    "checkpoint_path",
    "load_state_dict",
    "completed_trials",
    "evaluation_progress",
]

likely_evaluation_cells = []

for index in sorted(candidate_indices):
    source = sources[index]

    count = sum(
        indicator.lower() in source.lower()
        for indicator in evaluation_indicators
    )

    if count >= 2:
        likely_evaluation_cells.append(index)

print("\n" + "=" * 110)
print("LIKELY ORIGINAL VALIDATION EVALUATION CELLS")
print("=" * 110)
print(likely_evaluation_cells)

for index in likely_evaluation_cells:
    print("\n" + "=" * 110)
    print(f"COMPLETE VALIDATION-CANDIDATE SOURCE — CELL {index}")
    print("=" * 110)
    print(sources[index])


# ------------------------------------------------------------
# 6. Search for checkpoint-loading implementations
# ------------------------------------------------------------

checkpoint_patterns = [
    re.compile(r"torch\.load\s*\("),
    re.compile(r"load_state_dict\s*\("),
    re.compile(r"online_network_state_dict"),
]

checkpoint_cells = set()

for pattern in checkpoint_patterns:
    for index, source in enumerate(sources):
        if pattern.search(source):
            checkpoint_cells.add(index)

print("\n" + "=" * 110)
print("CHECKPOINT-LOADING SOURCE CELLS")
print("=" * 110)
print(sorted(checkpoint_cells))

# Print only cells that appear to combine checkpoint loading
# with validation evaluation.
for index in sorted(checkpoint_cells):
    source = sources[index]

    if (
        "validation" in source.lower()
        and (
            "evaluate_greedy_policy" in source
            or "per_sample_predictions" in source
            or "validation_evaluation" in source
        )
    ):
        print("\n" + "=" * 110)
        print(f"VALIDATION CHECKPOINT-LOADING CELL {index}")
        print("=" * 110)
        print(source)


# ------------------------------------------------------------
# 7. Safety summary
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("CELL 12 SAFETY SUMMARY")
print("=" * 110)
print("[PASS] Original notebook read as JSON.")
print("[PASS] Notebook cells were not executed.")
print("[PASS] No project modules imported.")
print("[PASS] No dataset arrays loaded.")
print("[PASS] No split-index values loaded.")
print("[PASS] No checkpoints loaded.")
print("[PASS] No validation or test policy evaluation performed.")
print("[PASS] No test samples accessed.")
print("[PASS] No project files modified.")

print("\nCell 12 completed. Review the helper and evaluator source.")

PHASE 4 — CELL 12: TRAINING HELPERS AND VALIDATION EVALUATOR AUDIT
Notebook: /home/ankit/Desktop/qskit/qec_ml/notebooks/RL_decoder_v2.ipynb
Total notebook cells: 160

----------------------------------------------------------------------------------------------------
DEFINITION SEARCH: select_experiment_action
Matching cells: [81]

----------------------------------------------------------------------------------------------------
DEFINITION SEARCH: double_dqn_update
Matching cells: [18]

----------------------------------------------------------------------------------------------------
DEFINITION SEARCH: make_ratio_dataset
Matching cells: [81]

----------------------------------------------------------------------------------------------------
DEFINITION SEARCH: ReplayBuffer
Matching cells: [17]

COMPLETE HELPER SOURCE — CELL 81
# Cell 70: Reproducible sampling-ratio experiment framework

import copy
import random
import numpy as np
import torch
import torch.nn.functional as F
from p

In [14]:
# ============================================================
# PHASE 4 — CELL 13
# Final source audit:
#   1. Fixed-budget training runner
#   2. 56-checkpoint audit / CELL78N_AUDIT_RECORDS
#   3. Exact greedy validation evaluator
#
# READ-ONLY SOURCE INSPECTION.
# ============================================================

from pathlib import Path
import json
import re

PROJECT_ROOT = Path("/home/ankit/Desktop/qskit/qec_ml")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "RL_decoder_v2.ipynb"

assert NOTEBOOK_PATH.is_file(), (
    f"Notebook not found: {NOTEBOOK_PATH}"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

def source_of(cell):
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source

sources = [source_of(cell) for cell in cells]

print("=" * 110)
print("PHASE 4 — CELL 13: FINAL TRAINING / CHECKPOINT / EVALUATOR SOURCE AUDIT")
print("=" * 110)
print("Notebook:", NOTEBOOK_PATH)
print("Total cells:", len(cells))


# ------------------------------------------------------------
# 1. Locate exact function and class definitions
# ------------------------------------------------------------

definition_names = [
    "run_fixed_budget_trial",
    "select_experiment_action",
    "double_dqn_update",
    "evaluate_greedy_policy",
    "QNetwork",
    "ReplayBuffer",
]

definition_cells = {}

for name in definition_names:
    pattern = re.compile(
        rf"^\s*(?:def|class)\s+{re.escape(name)}\s*[\(:]",
        re.MULTILINE,
    )

    matches = [
        i for i, source in enumerate(sources)
        if pattern.search(source)
    ]

    definition_cells[name] = matches

    print(f"\n{name}: {matches}")


# ------------------------------------------------------------
# 2. Print complete definitions
# ------------------------------------------------------------

printed = set()

for name in definition_names:
    for index in definition_cells[name]:
        if index in printed:
            continue

        printed.add(index)

        print("\n" + "=" * 110)
        print(f"COMPLETE DEFINITION CELL {index}")
        print("=" * 110)
        print(sources[index])


# ------------------------------------------------------------
# 3. Locate fixed-budget training runner call sites
# ------------------------------------------------------------

training_terms = [
    "run_fixed_budget_trial",
    "select_experiment_action",
    "double_dqn_update",
    "gradient_updates",
    "target_sync",
    "target_update",
    "training_progress.json",
    "checkpoint_path",
    "online_network_state_dict",
]

print("\n" + "=" * 110)
print("FIXED-BUDGET TRAINING SOURCE SEARCH")
print("=" * 110)

training_candidates = set()

for term in training_terms:
    matches = [
        i for i, source in enumerate(sources)
        if term.lower() in source.lower()
    ]

    print(f"{term}: {matches}")
    training_candidates.update(matches)


# Print cells likely to contain the actual runner or its caller.
training_indicators = [
    "run_fixed_budget_trial",
    "select_experiment_action",
    "double_dqn_update",
    "gradient_updates",
    "online_network_state_dict",
    "target_network_state_dict",
]

likely_training_cells = []

for index in sorted(training_candidates):
    source = sources[index]

    count = sum(
        term.lower() in source.lower()
        for term in training_indicators
    )

    if count >= 2:
        likely_training_cells.append(index)

print("\nLikely fixed-budget training cells:", likely_training_cells)

for index in likely_training_cells:
    if index in printed:
        continue

    printed.add(index)

    print("\n" + "=" * 110)
    print(f"COMPLETE FIXED-BUDGET TRAINING CELL {index}")
    print("=" * 110)
    print(sources[index])


# ------------------------------------------------------------
# 4. Locate the 56-checkpoint audit
# ------------------------------------------------------------

audit_terms = [
    "CELL78N_AUDIT_RECORDS",
    "checkpoint_audit",
    "checkpoint_sha256",
    "expected_sha256",
    "sha256",
    "replicate_08_50_50",
    "56 checkpoint",
    "checkpoint_manifest",
]

print("\n" + "=" * 110)
print("56-CHECKPOINT AUDIT SOURCE SEARCH")
print("=" * 110)

audit_candidates = set()

for term in audit_terms:
    matches = [
        i for i, source in enumerate(sources)
        if term.lower() in source.lower()
    ]

    print(f"{term}: {matches}")
    audit_candidates.update(matches)

audit_indicators = [
    "CELL78N_AUDIT_RECORDS",
    "sha256",
    "checkpoint",
    "trial_key",
    "replicate",
    "online_network_state_dict",
]

likely_audit_cells = []

for index in sorted(audit_candidates):
    source = sources[index]

    count = sum(
        term.lower() in source.lower()
        for term in audit_indicators
    )

    if count >= 3:
        likely_audit_cells.append(index)

print("\nLikely checkpoint-audit cells:", likely_audit_cells)

for index in likely_audit_cells:
    if index in printed:
        continue

    printed.add(index)

    print("\n" + "=" * 110)
    print(f"COMPLETE CHECKPOINT-AUDIT CELL {index}")
    print("=" * 110)
    print(sources[index])


# ------------------------------------------------------------
# 5. Locate exact greedy evaluation implementation and calls
# ------------------------------------------------------------

evaluator_terms = [
    "evaluate_greedy_policy",
    "detector_success",
    "logical_success",
    "correction_vectors",
    "correction_patterns",
    "terminated_by",
    "action_mask",
]

print("\n" + "=" * 110)
print("GREEDY EVALUATOR SOURCE SEARCH")
print("=" * 110)

evaluator_candidates = set()

for term in evaluator_terms:
    matches = [
        i for i, source in enumerate(sources)
        if term.lower() in source.lower()
    ]

    print(f"{term}: {matches}")
    evaluator_candidates.update(matches)

# The exact function-definition cells were printed above.
# Print additional likely evaluator implementation cells only.
evaluator_indicators = [
    "evaluate_greedy_policy",
    "detector_success",
    "logical_success",
    "correction_vectors",
    "terminated_by",
]

likely_evaluator_cells = []

for index in sorted(evaluator_candidates):
    source = sources[index]

    count = sum(
        term.lower() in source.lower()
        for term in evaluator_indicators
    )

    if count >= 3:
        likely_evaluator_cells.append(index)

print("\nLikely evaluator-related cells:", likely_evaluator_cells)

for index in likely_evaluator_cells:
    if index in printed:
        continue

    printed.add(index)

    print("\n" + "=" * 110)
    print(f"COMPLETE EVALUATOR-RELATED CELL {index}")
    print("=" * 110)
    print(sources[index])


# ------------------------------------------------------------
# 6. Explicitly search for test-set references in these cells
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("TEST-SET REFERENCES IN RELEVANT SOURCE CELLS")
print("=" * 110)

relevant_cells = (
    set(likely_training_cells)
    | set(likely_audit_cells)
    | set(likely_evaluator_cells)
)

test_terms = [
    "test_indices",
    "test_history",
    "test_final",
    "test_labels",
    "test_env",
    "test_set",
]

for index in sorted(relevant_cells):
    source = sources[index]

    found = [
        term for term in test_terms
        if term.lower() in source.lower()
    ]

    if found:
        print(f"Cell {index}: {found}")


# ------------------------------------------------------------
# 7. Safety summary
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("CELL 13 SAFETY SUMMARY")
print("=" * 110)
print("[PASS] Original notebook read as JSON.")
print("[PASS] Notebook cells were not executed.")
print("[PASS] No project modules imported.")
print("[PASS] No dataset arrays or split indices loaded.")
print("[PASS] No checkpoints loaded.")
print("[PASS] No model evaluation performed.")
print("[PASS] No test samples accessed.")
print("[PASS] No project files modified.")

print("\nCell 13 completed.")
print("Review the complete source definitions and call sites above.")

PHASE 4 — CELL 13: FINAL TRAINING / CHECKPOINT / EVALUATOR SOURCE AUDIT
Notebook: /home/ankit/Desktop/qskit/qec_ml/notebooks/RL_decoder_v2.ipynb
Total cells: 160

run_fixed_budget_trial: [85]

select_experiment_action: [81]

double_dqn_update: [18]

evaluate_greedy_policy: [28, 40, 46]

QNetwork: [16]

ReplayBuffer: [17]

COMPLETE DEFINITION CELL 85
# Cell 74: Fixed-gradient-update-budget Double DQN runner

import copy
import numpy as np
import torch


# ------------------------------------------------------------
# 1. Controlled experiment configuration
# ------------------------------------------------------------

FIXED_BUDGET_CONFIG = {
    **EXPERIMENT_CONFIG,
    "gradient_update_budget": 7000,
    "epsilon_decay_updates": 6000,
}

print("Fixed-budget experiment configuration:")
for key, value in FIXED_BUDGET_CONFIG.items():
    print(f"  {key}: {value}")


# ------------------------------------------------------------
# 2. Epsilon schedule based on gradient updates
# -----------

In [21]:

# ============================================================
# PHASE 4 — CELL 14
# COMPLETE CHECKPOINT HASH / MANIFEST VERIFICATION
#
# Original experiment : 21 trials
# Extension experiment: 35 trials
# Total               : 56 trials
#
# READ-ONLY AUDIT
# ============================================================

import json
import hashlib
from pathlib import Path


# ------------------------------------------------------------
# 1. EXPERIMENT DIRECTORIES
# ------------------------------------------------------------

ORIGINAL_DIR = Path(ORIGINAL_DIR).resolve()
EXTENSION_DIR = Path(EXTENSION_DIR).resolve()

assert ORIGINAL_DIR.is_dir(), (
    f"Original experiment directory missing: {ORIGINAL_DIR}"
)

assert EXTENSION_DIR.is_dir(), (
    f"Extension experiment directory missing: {EXTENSION_DIR}"
)


# ------------------------------------------------------------
# 2. SHA256 HELPER
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        while True:

            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. NORMALIZE CHECKPOINT RECORD FIELDS
# ------------------------------------------------------------

def get_checkpoint_filename(record):

    for key in (
        "checkpoint_file",
        "checkpoint_filename",
        "checkpoint_path",
        "file",
    ):

        value = record.get(key)

        if isinstance(value, str) and value.strip():

            return value.strip()

    raise AssertionError(
        f"{record.get('trial_key', 'Unknown trial')}: "
        "No checkpoint filename recorded."
    )


def get_checkpoint_hash(record):

    for key in (
        "checkpoint_sha256",
        "sha256",
        "checkpoint_hash",
    ):

        value = record.get(key)

        if isinstance(value, str) and len(value) == 64:

            return value.lower()

    raise AssertionError(
        f"{record.get('trial_key', 'Unknown trial')}: "
        "Missing or invalid checkpoint SHA256."
    )


# ------------------------------------------------------------
# 4. RESOLVE CHECKPOINT PATH
# ------------------------------------------------------------

def resolve_checkpoint_path(
    filename,
    base_directory,
    trial_key,
):

    base_directory = Path(base_directory).resolve()

    supplied_path = Path(filename)

    candidates = []

    # First try the exact recorded relative path.

    if supplied_path.is_absolute():

        candidates.append(supplied_path.resolve())

    else:

        candidates.append(
            (base_directory / supplied_path).resolve()
        )

        # The original experiment stores checkpoints
        # inside the checkpoints/ directory.

        candidates.append(
            (
                base_directory
                / "checkpoints"
                / supplied_path.name
            ).resolve()
        )

    # Search recursively by filename as a fallback.

    if not any(path.is_file() for path in candidates):

        candidates.extend(
            path.resolve()
            for path in base_directory.rglob(
                supplied_path.name
            )
            if path.is_file()
        )

    # Keep only existing files inside the experiment root.

    valid = sorted({
        path
        for path in candidates
        if path.is_file()
        and path.is_relative_to(base_directory)
    })

    if len(valid) == 0:

        raise FileNotFoundError(
            f"{trial_key}: checkpoint not found.\n"
            f"Recorded filename: {filename}\n"
            f"Experiment directory: {base_directory}"
        )

    if len(valid) > 1:

        raise AssertionError(
            f"{trial_key}: ambiguous checkpoint path.\n"
            + "\n".join(str(path) for path in valid)
        )

    return valid[0]


# ------------------------------------------------------------
# 5. VERIFY ONE EXPERIMENT
# ------------------------------------------------------------

def verify_checkpoint_records(
    records,
    base_directory,
    description,
):

    print("\n" + "=" * 88)
    print(f"{description} CHECKPOINT VERIFICATION")
    print("=" * 88)

    verified = {}

    for trial_key, record in records.items():

        assert isinstance(record, dict), (
            f"{trial_key}: invalid manifest record."
        )

        assert record.get("status", "completed") == "completed", (
            f"{trial_key}: trial is not marked completed."
        )

        filename = get_checkpoint_filename(record)

        expected_hash = get_checkpoint_hash(record)

        checkpoint_path = resolve_checkpoint_path(
            filename,
            base_directory,
            trial_key,
        )

        actual_hash = sha256_file(checkpoint_path)

        assert actual_hash == expected_hash, (
            f"\nSHA256 MISMATCH: {trial_key}\n"
            f"File: {checkpoint_path}\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )

        verified[trial_key] = {
            "checkpoint_path": checkpoint_path,
            "sha256": actual_hash,
            "size_bytes": checkpoint_path.stat().st_size,
        }

        print(
            f"[PASS] {trial_key}\n"
            f"       File: {checkpoint_path.relative_to(base_directory)}\n"
            f"       SHA256: {actual_hash}"
        )

    print("-" * 88)

    print("Completed records:", len(records))
    print("Verified checkpoints:", len(verified))

    assert len(verified) == len(records)

    return verified


# ------------------------------------------------------------
# 6. VERIFY ORIGINAL EXPERIMENT
# ------------------------------------------------------------

original_verified = verify_checkpoint_records(
    records=original_trials,
    base_directory=ORIGINAL_DIR,
    description="ORIGINAL",
)


# ------------------------------------------------------------
# 7. VERIFY EXTENSION EXPERIMENT
# ------------------------------------------------------------

extension_verified = verify_checkpoint_records(
    records=extension_trials,
    base_directory=EXTENSION_DIR,
    description="EXTENSION",
)


# ------------------------------------------------------------
# 8. GLOBAL CONSISTENCY CHECKS
# ------------------------------------------------------------

assert len(original_verified) == 21

assert len(extension_verified) == 35

all_verified = {
    **original_verified,
    **extension_verified,
}

assert len(all_verified) == 56

assert (
    set(original_verified)
    .isdisjoint(set(extension_verified))
), "Duplicate trial keys across experiments."


# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("PHASE 4 — CELL 14: FINAL AUDIT SUMMARY")
print("=" * 88)

print("Original completed trials :", len(original_trials))
print("Original verified         :", len(original_verified))

print("Extension completed trials:", len(extension_trials))
print("Extension verified        :", len(extension_verified))

print("-" * 88)

print("Total completed trials    :", 56)
print("Total verified checkpoints:", len(all_verified))

print("\nINTEGRITY RESULTS")

print("[PASS] Original checkpoint hashes verified.")
print("[PASS] Extension checkpoint hashes verified.")
print("[PASS] All 56 checkpoint files resolved.")
print("[PASS] All recorded SHA256 hashes match.")
print("[PASS] No duplicate trial keys.")
print("[PASS] All checkpoint paths remain inside their experiment directories.")

print("\nSAFETY RESULTS")

print("[PASS] No checkpoint files modified.")
print("[PASS] No manifest files modified.")
print("[PASS] No training performed.")
print("[PASS] No validation evaluation performed.")
print("[PASS] No test samples accessed.")

print("\n" + "=" * 88)
print("CHECKPOINT MANIFEST VERIFICATION COMPLETED")
print("=" * 88)


ORIGINAL CHECKPOINT VERIFICATION
[PASS] replicate_01_natural
       File: checkpoints/replicate_01_natural.pt
       SHA256: e7100683a1b71e929852767bc5af6708e403ce95559808e6fc1248ba660d668c
[PASS] replicate_01_80_20
       File: checkpoints/replicate_01_80_20.pt
       SHA256: 93b1cd4d20a667ef9716e7469f622be50fd8c8e879ad5d6b5d68cf55466533f5
[PASS] replicate_01_75_25
       File: checkpoints/replicate_01_75_25.pt
       SHA256: 694ee4302acab9b6d57e5320ca101d71da18bcac5fc4723356b58dc0933b8ea0
[PASS] replicate_01_70_30
       File: checkpoints/replicate_01_70_30.pt
       SHA256: 5894ba32f236dff20c9e2e92f09237959b9251c57fe8c392a4dd8ed0e58aa419
[PASS] replicate_01_65_35
       File: checkpoints/replicate_01_65_35.pt
       SHA256: a0f84dfaf445da35296962c6a4598249e912c289371861b6b9937ae1a5b39150
[PASS] replicate_01_60_40
       File: checkpoints/replicate_01_60_40.pt
       SHA256: 50dbb518b399a007dae9de074ae11313a8cb2a5a1d3b6cdb595bf526d264a854
[PASS] replicate_01_50_50
       File: check

In [22]:

# ============================================================
# PHASE 4 — NOTEBOOK STRUCTURE INSPECTION
# READ-ONLY
# ============================================================

from pathlib import Path

project_dir = Path.cwd()

print("=" * 75)
print("PROJECT DIRECTORY")
print("=" * 75)
print(project_dir)

print("\n" + "=" * 75)
print("NOTEBOOK FILES")
print("=" * 75)

for path in sorted(project_dir.rglob("*.ipynb")):
    print(path.relative_to(project_dir))

print("\n" + "=" * 75)
print("PROJECT SUBDIRECTORIES")
print("=" * 75)

for path in sorted(project_dir.iterdir()):
    if path.is_dir():
        print(path.name)

print("\nINSPECTION COMPLETED — NO FILES MODIFIED")


PROJECT DIRECTORY
/home/ankit/Desktop/qskit/qec_ml/notebooks

NOTEBOOK FILES
.ipynb_checkpoints/RL_agent_-checkpoint.ipynb
.ipynb_checkpoints/RL_decoder_v2-checkpoint.ipynb
.ipynb_checkpoints/RL_decoder_vectorized-checkpoint.ipynb
.ipynb_checkpoints/data_generation_first_mlp-checkpoint.ipynb
.ipynb_checkpoints/final_test_evaluation-checkpoint.ipynb
.ipynb_checkpoints/mlp_v2_vs_pymatching-checkpoint.ipynb
RL_agent_.ipynb
RL_decoder_v2.ipynb
RL_decoder_vectorized.ipynb
data_generation_first_mlp.ipynb
final_test_evaluation.ipynb
mlp_v2_vs_pymatching.ipynb

PROJECT SUBDIRECTORIES
.ipynb_checkpoints
phase2_freeze_20261104
phase4_final_test_results
qec_experiment_results
qec_model_checkpoints

INSPECTION COMPLETED — NO FILES MODIFIED


In [23]:

from pathlib import Path
import json

# ------------------------------------------------------------
# 1. PROJECT PATH
# ------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name != "notebooks":
    raise RuntimeError(
        "Please run this cell from the project's notebooks directory.\n"
        f"Current directory: {NOTEBOOK_DIR}"
    )

RL_NOTEBOOKS = [
    "RL_agent_.ipynb",
    "RL_decoder_v2.ipynb",
    "RL_decoder_vectorized.ipynb",
]

KEYWORDS = [
    "stim.Circuit",
    "Circuit.generated",
    "rotated_memory",
    "depolarization",
    "flip_probability",
    "detector",
    "observable",
    "compile_detector_sampler",
    "detector_sampler",
    "sample(",
    "logical",
    "noise",
    "shots",
]

# ------------------------------------------------------------
# 2. READ-ONLY NOTEBOOK INSPECTION
# ------------------------------------------------------------

print("=" * 90)
print("RL NOTEBOOK SOURCE INSPECTION")
print("=" * 90)
print("Notebook directory:", NOTEBOOK_DIR)

for notebook_name in RL_NOTEBOOKS:

    notebook_path = NOTEBOOK_DIR / notebook_name

    print("\n" + "=" * 90)
    print("NOTEBOOK:", notebook_name)
    print("=" * 90)

    if not notebook_path.is_file():
        print("[NOT FOUND]")
        continue

    with notebook_path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    cells = notebook.get("cells", [])
    matched_cells = []

    for index, cell in enumerate(cells):

        source = "".join(cell.get("source", []))
        source_lower = source.lower()

        matches = [
            keyword
            for keyword in KEYWORDS
            if keyword.lower() in source_lower
        ]

        if matches:
            matched_cells.append(
                (index, cell.get("cell_type", "unknown"), matches, source)
            )

    print("Total cells:", len(cells))
    print("Relevant cells:", len(matched_cells))

    for index, cell_type, matches, source in matched_cells:

        print("\n" + "-" * 90)
        print(f"CELL INDEX: {index}")
        print("CELL TYPE:", cell_type)
        print("MATCHED KEYWORDS:", ", ".join(matches))
        print("-" * 90)

        print(source if source.strip() else "[EMPTY CELL]")

print("\n" + "=" * 90)
print("INSPECTION COMPLETED — NO FILES MODIFIED")
print("=" * 90)


RL NOTEBOOK SOURCE INSPECTION
Notebook directory: /home/ankit/Desktop/qskit/qec_ml/notebooks

NOTEBOOK: RL_agent_.ipynb
Total cells: 55
Relevant cells: 45

------------------------------------------------------------------------------------------
CELL INDEX: 0
CELL TYPE: code
MATCHED KEYWORDS: logical, noise
------------------------------------------------------------------------------------------
'''
$$ \boxed{\text{QEC environment} \rightarrow \text{RL decoder} \rightarrow \text{baseline comparison} \rightarrow \text{scaling}} $$
The agent observes syndrome information and learns a policy for applying corrections.

For a surface code:

$$ \text{noise} \rightarrow \text{syndrome} \rightarrow \boxed{\text{RL agent}} \rightarrow \text{correction} \rightarrow \text{logical success/failure}. $$

We can then compare:
    Majority baseline
    PyMatching
    Supervised MLP
    RL decoder

Using: $$ d=3,\;5,\;7 $$ but not necessarily train everything from scratch independently.

A particular